# Experiment: SRM Training Workflow

Objective:
- Build grouped splits and per-method SRM training job plan.
- Define SRM feature-extraction + fold/method training loop structure.
- Keep the notebook runnable before SRM is implemented.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


def read_rows_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', newline='', encoding='utf-8') as f:
        return [dict(r) for r in csv.DictReader(f)]


def count_rows(path: Path) -> int:
    return len(read_rows_csv(path))


## Parameters

In [ ]:
from src.pipeline.config import PipelineConfig
from src.pipeline.runner import PipelineRunner
from src.detection.srm import (
    SRMTrainingInput,
    extract_srm_features,
    train_srm_ec_model,
    score_srm_ec_model,
)

COVERS_MASTER = PROJECT_ROOT / 'data/manifests/covers_master.csv'
SIMULATE_UNTIL_SRM_IS_IMPLEMENTED = True


## Build Splits and Training Jobs

In [ ]:
if not COVERS_MASTER.exists():
    raise FileNotFoundError(
        f'Missing {COVERS_MASTER}. Run data generation/merge first (or notebook 02).'
    )

cfg = PipelineConfig.from_project_root(PROJECT_ROOT)
runner = PipelineRunner(cfg)
runner.init_layout()

splits_path = runner.create_grouped_splits(covers_manifest_path=COVERS_MASTER)
jobs_path = runner.build_srm_training_jobs(splits_json_path=splits_path)

print(f'Splits JSON: {splits_path}')
print(f'SRM jobs CSV: {jobs_path}')
print(f'Number of SRM jobs: {count_rows(jobs_path)}')


In [ ]:
jobs = read_rows_csv(PROJECT_ROOT / 'results/splits/srm_training_jobs.csv')
print('First 5 jobs:')
for row in jobs[:5]:
    print(row)

by_method: dict[str, int] = {}
for row in jobs:
    by_method[row['method']] = by_method.get(row['method'], 0) + 1
print(f'Jobs by method: {by_method}')


## Training Loop Template (Per Fold, Per Method)

This is the function-calling structure your SRM implementation should satisfy.

Deferred SRM functions in this workflow: `extract_srm_features`, `train_srm_ec_model`, `score_srm_ec_model`.


In [ ]:
# Real training data wiring template (to enable once feature extraction is implemented):
# from src.data.images import load_image
# def extract_feature_matrix(image_paths: list[Path]) -> list[list[float]]:
#     return [extract_srm_features(load_image(p)) for p in image_paths]

def build_dummy_training_input(method: str, fold: int) -> SRMTrainingInput:
    # Replace this with real SRM feature extraction from train/val split images.
    x_train = [[0.0, 0.1], [0.8, 0.9], [0.2, 0.3], [0.7, 0.8]]
    y_train = [0, 1, 0, 1]
    x_val = [[0.1, 0.2], [0.9, 0.8]]
    y_val = [0, 1]
    return SRMTrainingInput(
        method=method,
        fold=fold,
        x_train=x_train,
        y_train=y_train,
        x_val=x_val,
        y_val=y_val,
        random_seed=42,
    )

training_status: list[dict[str, object]] = []
for row in jobs:
    fold = int(row['fold'])
    method = row['method']

    if SIMULATE_UNTIL_SRM_IS_IMPLEMENTED:
        training_status.append({'fold': fold, 'method': method, 'status': 'planned'})
        continue

    training_input = build_dummy_training_input(method=method, fold=fold)
    model = train_srm_ec_model(training_input)
    test_scores = score_srm_ec_model(model, x_samples=[[0.15, 0.2], [0.85, 0.9]])
    training_status.append(
        {
            'fold': fold,
            'method': method,
            'status': 'trained',
            'n_scores': len(test_scores),
        }
    )

print('Training status summary (first 10 rows):')
for row in training_status[:10]:
    print(row)


## Notes

- When SRM is implemented, set `SIMULATE_UNTIL_SRM_IS_IMPLEMENTED=False`.
- Keep models per `(method, fold)` to match the locked 10-run design.
- Persist any model artifacts outside deferred functions (pipeline layer owns writes).
- Once SRM deferred functions are implemented, `run_detector_stage` can auto-train SRM per `(fold, method)` during detector execution.
